# C, D. 시장성 · 이해관계자 평가 에이전트

| | C. 시장성 | D. 이해관계자 |
|---|---|---|
| **담당** | Web Search | Web Search |
| **선행 노드** | B | B |
| **출력** | `market_eval`, `market_references` | `stakeholder_eval`, `stakeholder_references` |

둘 다 RAG 없이 웹 검색만 쓰고, `tech_research`(B의 결과)를 참고해 검색어를 구체화한다는 점이 같아서 한 노트북에 같이 둔다.
시장성은 "문서·저장소·규격의 기록"만, 이해관계자는 "발화"만 본다 — 겹치지 않게 역할이 나뉜다(2-2/2-3절).

이 노트북 끝에서 만든 함수 둘은 `src/nodes_cd.py`로 저장된다.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import prompts
from src.node_utils import run_web_search, summarize_tech_research
from src.schemas import MarketEval, StakeholderEval

## 1. 프롬프트 확인

In [ ]:
print(prompts.MARKET_EVAL_PROMPT)
print("\n" + "="*80 + "\n")
print(prompts.STAKEHOLDER_EVAL_PROMPT)

## 2. 노드 함수 정의

In [ ]:
def make_node_c(llm, web_search_tool):
    """C. 시장성 평가."""
    structured_llm = llm.with_structured_output(MarketEval)

    def node_c_market_eval(state):
        tech_context = summarize_tech_research(state.get("tech_research", {}))
        search_results = run_web_search(
            web_search_tool,
            [
                "TurboQuant KV cache quantization adoption production",
                "InfiniGen KV cache offloading adoption vLLM llama.cpp",
            ],
        )
        prompt = prompts.MARKET_EVAL_PROMPT.format(
            tech_context=tech_context, search_results=search_results
        )
        result = structured_llm.invoke(prompt)
        refs = [{"source": "web_search", "detail": search_results[:200]}]
        return {"market_eval": result.model_dump(), "market_references": refs}

    return node_c_market_eval

In [ ]:
def make_node_d(llm, web_search_tool):
    """D. 이해관계자 평가."""
    structured_llm = llm.with_structured_output(StakeholderEval)

    def node_d_stakeholder_eval(state):
        tech_context = summarize_tech_research(state.get("tech_research", {}))
        search_results = run_web_search(
            web_search_tool,
            [
                "TurboQuant developer opinion review",
                "InfiniGen competing method comparison criticism",
            ],
        )
        prompt = prompts.STAKEHOLDER_EVAL_PROMPT.format(
            tech_context=tech_context, search_results=search_results
        )
        result = structured_llm.invoke(prompt)
        refs = [{"source": "web_search", "detail": search_results[:200]}]
        return {"stakeholder_eval": result.model_dump(), "stakeholder_references": refs}

    return node_d_stakeholder_eval

## 3. 배선 테스트 — API 키 없이

In [ ]:
from src.schemas import TechStatus

class FakeStructuredLLM:
    def __init__(self, output):
        self.output = output
    def invoke(self, prompt):
        return self.output

class FakeLLM_C:
    def with_structured_output(self, schema_cls):
        ts = TechStatus(turboquant="정식", infinigen="실험")
        return FakeStructuredLLM(MarketEval(
            market_size_growth="추정 갈림", adoption_status=ts, ecosystem_support=ts,
            standardization="있음", label="조건 의존", notes="가짜",
        ))

class FakeLLM_D:
    def with_structured_output(self, schema_cls):
        ts = TechStatus(turboquant="채택했다고 말함", infinigen="조건부")
        return FakeStructuredLLM(StakeholderEval(
            competing_camp_reaction=ts, developer_adoption=ts, investor_coverage=ts,
            label="조건 의존", notes="가짜",
        ))

class FakeWebSearchTool:
    def invoke(self, args):
        return f"가짜 검색 결과: {args['query']}"

sample_state = {"tech_research": {"TurboQuant": {"overview": "가짜 개요"}}}

node_c = make_node_c(FakeLLM_C(), FakeWebSearchTool())
result_c = node_c(sample_state)
assert result_c["market_eval"]["label"] == "조건 의존"
print("C 배선 OK:", result_c["market_eval"]["adoption_status"])

node_d = make_node_d(FakeLLM_D(), FakeWebSearchTool())
result_d = node_d(sample_state)
assert result_d["stakeholder_eval"]["label"] == "조건 의존"
print("D 배선 OK:", result_d["stakeholder_eval"]["developer_adoption"])

## 4. 실제 LLM 테스트

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("OPENAI_API_KEY") and os.environ.get("TAVILY_API_KEY"):
    from langchain.chat_models import init_chat_model
    from langchain_tavily import TavilySearch
    from src import config

    real_llm = init_chat_model(config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=0)
    real_web_search = TavilySearch(max_results=5)

    node_c_real = make_node_c(real_llm, real_web_search)
    print(node_c_real(sample_state)["market_eval"])
else:
    print("API 키 없음 - 이 셀은 건너뜀.")

## 5. 파일로 저장

In [ ]:
import inspect

q3 = chr(34) * 3
with open("../src/nodes_cd.py", "w", encoding="utf-8") as f:
    f.write(q3 + "C, D. 시장성/이해관계자 노드 - 02_agent_CD_market_stakeholder.ipynb에서 생성됨.\n")
    f.write("이 파일을 직접 고치지 말고, 노트북에서 고친 뒤 저장 셀을 다시 실행할 것." + q3 + "\n\n")
    f.write("from src import prompts\n")
    f.write("from src.node_utils import run_web_search, summarize_tech_research\n")
    f.write("from src.schemas import MarketEval, StakeholderEval\n\n\n")
    f.write(inspect.getsource(make_node_c))
    f.write("\n\n")
    f.write(inspect.getsource(make_node_d))
    f.write("\n")

print("src/nodes_cd.py 저장 완료")